In [7]:
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델 객체 생성 (가성비가 좋은 small 모델 사용)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 텍스트 임베딩 수행
query_result = embeddings.embed_query('인공지능')

print(f"벡터의 길이(차원 수): {len(query_result)}")
print(f"변환된 벡터의 앞부분 5개 값: {query_result[:5]}")


벡터의 길이(차원 수): 1536
변환된 벡터의 앞부분 5개 값: [-0.01385547686368227, 0.0200613122433424, 0.007972599007189274, 0.007269692607223988, -0.01721169427037239]


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

store = LocalFileStore("./cache/")

# 시를 사용할 **준비(설정)**만 마친 상태이기 때문에,
# 실제로 임베딩 작업을 수행하는 명령을 내리기 전까지는
# ./cache/ 폴더에 아무런 파일도 생성되지 않습니다.

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace=underlying_embeddings.model
)


`CacheBackedEmbeddings` (캐시 임베더)의 핵심 역할은 **"이미 계산한 임베딩 값을 재사용하여 돈(API 비용)과 시간(대기 시간)을 아끼는 것"**입니다.

구체적으로 어떤 일을 하는지 단계별로 설명해 드릴게요.

### 1. 작동 원리 (비유)
임베딩은 텍스트를 숫자로 바꾸는 과정인데, OpenAI 같은 API를 쓸 때마다 비용이 발생합니다.
- **캐시 없음:** 똑같은 문장을 100번 물어보면, OpenAI에게 100번 돈을 내고 숫자로 바꿔달라고 시킵니다.
- **캐시 있음:** 
    1. 첫 번째 질문: OpenAI에게 돈을 내고 숫자를 받아온 뒤, **`./cache/` 폴더에 저장**해둡니다.
    2. 두 번째 질문 (똑같은 문장): OpenAI에게 가지 않고, **내 컴퓨터의 `./cache/` 폴더**에서 저장된 숫자를 꺼내옵니다.

### 2. 코드에서의 상세 역할

```python:15:19:04/04.ipynb
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings, # 실제 임베딩을 수행할 모델 (OpenAI)
    store,                 # 임베딩 결과를 저장할 장소 (로컬 폴더)
    namespace=underlying_embeddings.model # 모델별로 구분해서 저장 (예: text-embedding-3-small)
)
```

*   **중복 계산 방지:** 문서가 수천 페이지일 때, 내용을 조금만 수정하고 다시 임베딩을 돌려도 **바뀌지 않은 부분은 캐시에서 즉시 가져옵니다.**
*   **비용 절감:** OpenAI API 호출 횟수를 획기적으로 줄여줍니다.
*   **속도 향상:** 네트워크 통신(API 호출)보다 내 컴퓨터의 파일을 읽는 것이 훨씬 빠릅니다.

### 3. 어떻게 똑같은 문장인지 아나요?
주석(@04.ipynb 9-13라인)에 잘 설명되어 있듯이, **해시(Hash)** 기술을 사용합니다.
- 문장의 내용을 복잡한 문자열(해시값)로 바꿉니다.
- 내용이 글자 하나라도 다르면 해시값이 완전히 달라지므로, **내용이 100% 똑같을 때만** 캐시를 사용합니다.

### 요약
- **목적:** **비용 절약 + 속도 향상**
- **방법:** 한 번 계산한 임베딩 결과를 하드디스크(`./cache/`)에 저장해두고, 똑같은 문장이 들어오면 다시 계산하지 않고 꺼내 씀.

RAG 시스템을 개발할 때, 특히 문서의 양이 많거나 테스트를 자주 해야 할 때 **필수적으로 사용하는 최적화 기법**입니다.

-------

In [11]:
import time

# 테스트할 텍스트 리스트 
test_texts = ["안녕하세요 " + str(i) for i in range(10)]

# --- 1. 첫 번째 실행 (캐시가 생성되는 시점) ---
print("1차 실행 시작 (OpenAI 호출 예상)...")
start_time = time.time()

# 임베딩 수행
cached_embedder.embed_documents(test_texts)

end_time = time.time()
print(f"1차 실행 소요 시간: {end_time - start_time:.4f}초")
print("-" * 30)

# --- 2. 두 번째 실행 (캐시를 사용하는 시점) ---
print("2차 실행 시작 (로컬 캐시 사용 예상)...")
start_time = time.time()

# 똑같은 텍스트로 다시 임베딩 수행
cached_embedder.embed_documents(test_texts)

end_time = time.time()
print(f"2차 실행 소요 시간: {end_time - start_time:.4f}초")

1차 실행 시작 (OpenAI 호출 예상)...
1차 실행 소요 시간: 0.0107초
------------------------------
2차 실행 시작 (로컬 캐시 사용 예상)...
2차 실행 소요 시간: 0.0043초


-------

### 1. 캐시 임베더 (CacheBackedEmbeddings): "이미 계산한 건 또 하지 말자"
임베딩(문장을 숫자로 바꾸는 것)은 **비싼 작업**입니다. 
- **문제:** 똑같은 문장을 임베딩할 때마다 OpenAI에 돈을 내고 기다려야 합니다.
- **해결:** 한 번 계산한 결과는 내 컴퓨터 하드디스크(`store`)에 저장해둡니다. 다음에 똑같은 문장이 오면 OpenAI에 안 가고 하드디스크에서 꺼내 씁니다.
- **`namespace`**: "이건 GPT-4용 임베딩이야", "이건 Gemini용이야"라고 포스트잇을 붙여서 데이터가 섞이지 않게 관리하는 것입니다.

---

### 2. 벡터 스토어 (Vector Store): "수만 개의 데이터 중 정답을 빛의 속도로 찾자"
임베딩을 마친 데이터는 수천 개의 숫자로 이루어진 **벡터(Vector)** 형태입니다. 
- **문제:** 문서가 10만 개라면, 사용자의 질문과 10만 개의 벡터를 하나하나 일일이 비교(수학적 계산)하는 것은 너무 느립니다. 일반적인 엑셀이나 DB로는 이 계산을 감당하기 힘듭니다.
- **해결:** **벡터 전용 데이터베이스(Vector Store)**를 사용합니다. 
    - 벡터 스토어는 고차원 데이터를 **검색하기 좋게 특수한 구조**로 저장합니다. 
    - 질문이 들어오면 10만 개를 다 뒤지는 게 아니라, 수학적인 지름길을 통해 **가장 비슷한 문서 몇 개를 순식간에** 찾아냅니다.

---

### 3. 전체 흐름 요약 (비유)

1.  **문서 로드 & 쪼개기:** 두꺼운 책을 한 페이지씩 찢어서 준비합니다.
2.  **임베딩 & 캐싱:** 각 페이지의 내용을 숫자로 바꿉니다. (이때, 이미 숫자로 바꾼 페이지는 캐시에서 가져와서 **돈과 시간을 아낍니다.**)
3.  **벡터 스토어 저장:** 이 숫자들을 벡터 스토어라는 **'특수 도서관'**에 꽂아둡니다.
4.  **검색:** 사용자가 질문하면, 벡터 스토어 도서관에서 질문의 숫자와 가장 비슷한 페이지를 **빛의 속도로** 찾아줍니다.

### 결론
- **캐시 임베더**는 **"임베딩 과정 자체"**의 중복을 막아 **비용**을 아껴주는 것이고,
- **벡터 스토어**는 **"임베딩된 데이터들 사이에서 검색"**을 **속도** 있게 처리해주는 전용 저장소입니다.

이 두 가지가 합쳐져야 수많은 문서 데이터 속에서도 빠르고 저렴하게 답변을 찾아주는 RAG 시스템이 완성됩니다.

-------

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PDFPlumberLoader


loader = PDFPlumberLoader("./pdf-file.pdf")

docs = loader.load()


text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)

recursive_docs = text_splitter.split_documents(docs)

vectorstore = InMemoryVectorStore.from_documents(
    documents=recursive_docs,
    embedding=cached_embedder
)

# cache 폴더를 보면 파일이 생성된 것 확인할 수 있다
# 해당 파일은 임베딩 결과를 저장하는 파일로, recursive_docs의 page_content를 임베딩 한 결과를 저장하는 파일이다
# 파일 이름은 임베딩 모델의 이름과 page_content의 해시값으로 구성된다
# 띠리사 page_content와 똑같은 입력이 들어오면, 해싱을 한 다음 
# 동일한 파일이 있는지 찾고, 만약 같은게 있다면 임베딩 모델에 입력이 전달되는 것이 아니라
# 찾은 폴더의 파일을 읽어서 임베딩 결과를 반환하는 것이다



### 1. 작동 원리: "문장 단위"가 아닌 "청크(Chunk) 단위" 캐싱
`text_splitter`를 통해 원본 PDF를 여러 개의 작은 `recursive_docs`로 쪼개셨죠? 캐시 임베더는 **이 쪼개진 조각(Chunk) 하나하나의 전체 내용**을 기준으로 캐싱을 수행합니다.

1.  **해시값 생성:** 각 청크의 `page_content`(텍스트 내용) 전체를 해싱(Hashing)하여 고유한 ID를 만듭니다.
2.  **파일 확인:** `./cache/` 폴더에 그 ID와 일치하는 파일이 있는지 확인합니다.
3.  **판단:**
    *   **파일이 없다면:** OpenAI API를 호출하여 임베딩 값을 받아온 뒤, `./cache/`에 저장합니다.
    *   **파일이 있다면:** OpenAI에 가지 않고, **파일에 저장된 숫자(임베딩 값)를 즉시 읽어옵니다.**

---

### 2. 질문하신 "같은 문장이나 단어"의 경우

*   **청크 내용이 100% 일치할 때:**
    만약 PDF 내에 완전히 똑같은 내용의 청크가 두 번 나타난다면(예: "제1장 개요"라는 청크가 여러 번 반복됨), **두 번째부터는 임베딩을 다시 하지 않습니다.** 캐시에서 가져옵니다.

*   **청크 내용 중 일부 단어만 같을 때:**
    캐시는 **청크 전체 텍스트**를 기준으로 작동합니다. 
    - 청크 A: "사과는 맛있다."
    - 청크 B: "사과는 빨갛다."
    두 청크 모두 "사과는"이라는 단어가 들어있지만, **전체 텍스트는 다르기 때문에** 캐시 임베더는 이 둘을 완전히 다른 데이터로 취급합니다. 즉, 둘 다 각각 임베딩을 수행합니다.

---

### 3. 실제 개발 시 가장 큰 이점 (재실행 시)

이 캐싱의 진짜 위력은 **코드를 다시 실행할 때** 나타납니다.

1.  **첫 번째 실행:** PDF를 읽고 100개의 청크로 쪼개어 임베딩합니다. (OpenAI API 100번 호출, 비용 발생)
2.  **두 번째 실행 (코드 수정 후):** 검색 로직을 고치기 위해 코드를 다시 돌립니다. 이때 PDF 내용은 그대로라면, **100개 청크 모두 캐시에서 즉시 읽어옵니다.** (OpenAI 호출 0번, 비용 0원, 속도 매우 빠름!)

### 요약
- **캐싱 기준:** 쪼개진 **청크(Chunk) 전체 텍스트**가 100% 일치해야 합니다.
- **단어 단위:** 단어 하나하나를 캐싱하는 것이 아니라, **청크 덩어리**를 통째로 캐싱합니다.
- **효과:** 똑같은 문서를 다시 처리하거나, 문서 내에 중복된 내용이 있을 때 **비용과 시간을 획기적으로 아껴줍니다.**

-----

PDF 파일의 일부 내용만 수정되었을 때, 캐시 임베더(`CacheBackedEmbeddings`)가 어떻게 똑똑하게 대처하는지 설명해 드릴게요.

결론부터 말씀드리면, **"수정된 부분만 새로 임베딩하고, 수정되지 않은 나머지 부분은 캐시에서 그대로 가져옵니다."**

---

### 1. 작동 과정 (예시)

원본 PDF가 10페이지이고, 이를 쪼개서 총 **100개의 청크(Chunk)**가 만들어졌다고 가정해 봅시다.

1.  **첫 번째 실행:** 100개 청크 모두 임베딩하여 `./cache/`에 저장합니다.
2.  **PDF 수정:** 5페이지의 오타 한 글자를 고쳤습니다.
3.  **두 번째 실행:**
    *   `text_splitter`가 다시 PDF를 쪼갭니다.
    *   수정되지 않은 1~4페이지, 6~10페이지에서 나온 **90개 청크**는 이전과 내용이 100% 똑같습니다. -> **캐시에서 즉시 로드 (비용 0)**
    *   수정된 5페이지 근처에서 나온 **10개 청크**는 내용이 바뀌었습니다. -> **OpenAI API 호출하여 새로 임베딩 (비용 발생)**

---

### 2. 왜 이렇게 효율적인가요?

*   **청크 단위의 독립성:** 캐시는 파일 전체가 아니라 **쪼개진 조각(Chunk) 하나하나를 기준**으로 파일을 만듭니다.
*   **해시값의 특징:** 내용이 조금이라도 바뀌면 해당 조각의 해시값만 바뀌기 때문에, 다른 조각들의 캐시 파일에는 영향을 주지 않습니다.

---

### 3. 주의할 점 (Chunking의 영향)

만약 PDF 앞부분에 내용을 추가해서 **청크가 밀려난다면** 어떻게 될까요?

*   내용은 같아도 쪼개지는 지점이 달라지면(예: 이전에는 "A/B"로 쪼개졌는데 이번에는 "AB/"로 쪼개짐), 텍스트 내용 자체가 달라진 것으로 인식되어 **다시 임베딩**을 하게 됩니다.
*   하지만 `RecursiveCharacterTextSplitter`는 최대한 의미 단위로 쪼개려 노력하므로, 수정된 부분 이후의 청크들이 운 좋게 이전과 똑같이 쪼개진다면 여전히 캐시를 활용할 수 있습니다.

### 요약
- **수정된 조각만 새로 계산**하고, 나머지는 재사용합니다.
- 덕분에 대용량 문서를 다룰 때 **수정 사항이 생겨도 전체를 다시 임베딩할 필요가 없어** 매우 경제적입니다.

------

In [6]:
query = "Tesla 투자 비중이 얼마나 되나요?"
results = vectorstore.similarity_search(query)
# 벡터스토어에 있는 문서와 유사도를 계산하고, 유사도가 높은 순서대로 검색 결과를 반환한다

print(f"검색된 문서 내용:\n{results[0].page_content}")
# 가장 유사한 문서의 내용을 반환한다
# 이 내용을 LLM에게 전달해서 답변을 생성할 수 있다
# 이것이 RAG 

검색된 문서 내용:
Holdings Data - ARKK
As of 11/26/2025
ARKK
ARK Innovation ETF
Company Ticker CUSIP Shares Market Value ($) Weight (%)
1 TESLA INC TSLA 88160R101 2,204,438 $924,541,297.20 12.26%
2 TEMPUS AI INC TEM 88023B103 5,465,331 $419,956,034.04 5.57%
3 ROKU INC ROKU 77543R102 4,347,025 $412,445,732.00 5.47%
